# ResLSTM-SER: Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection

Reproducible training notebook for the DSPA 2026 paper:
**"Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection"**
by Daniil Krasnoproshin and Maxim Vashkevich.

> IEEE Xplore: https://ieeexplore.ieee.org/document/11476771

This notebook performs:
1. Feature extraction (MFCC + chromagram) from RAVDESS .wav files
2. Global mean/std normalization
3. Model training with Optuna hyperparameter optimization
4. 10-run statistical evaluation with optimal hyperparameters
5. K-fold cross-validation (speaker-independent, 5-fold)

In [1]:
import torch
print(f"CUDA có khả dụng không?: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

CUDA có khả dụng không?: True
Tên GPU: NVIDIA GeForce RTX 2060 SUPER


## Imports & Path Configuration

In [2]:
import glob
import os
import sys
from typing import List, Tuple

import numpy as np
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from ignite.metrics.recall import Recall
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from tqdm import tqdm

# ── Paths — ADJUST THESE ──
# The RAVDESS dataset is NOT included in this repository.
# Download from: https://www.kaggle.com/datasets/uwrfkaggler/ravdess-emotional-speech-audio
# or: https://zenodo.org/record/1188976

ROOT_DIR = os.path.abspath('')  # current working directory
RAVDESS_DATA_PATH = os.path.join(ROOT_DIR, 'data', 'ravdess', 'audio_speech_actors_01-24', '**', '*.wav')
EXTRACTED_DATA_DIR = os.path.join(ROOT_DIR, 'data', 'ravdess', 'extracted')
MODEL_BACKUP_DIR = os.path.join(ROOT_DIR, 'model_backup')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
print(f"RAVDESS path: {RAVDESS_DATA_PATH}")

Device: cuda
RAVDESS path: d:\Speech_Processing\ResLSTM-SER_with_wav2vec\data\ravdess\audio_speech_actors_01-24\**\*.wav


d:\Speech_Processing\ResLSTM-SER_with_wav2vec\.venv\Lib\site-packages\ignite\handlers\checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


## Reproducibility

In [3]:
def set_seed(seed: int = 42):
    """Set random seed for reproducibility."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## RAVDESS Label Mappings

In [4]:
def get_emotions():
    return ["neutral", "calm", "happy", "sad", "angry", "fear", "disgust", "surprised"]

def get_emotions_dictionary():
    return {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fear",
        "07": "disgust",
        "08": "surprised",
    }

def get_actors():
    return {f"{i:02d}": str(i) for i in range(1, 25)}

## Feature Extraction (wav → npy)

In [5]:
def extract_features(file_path: str, processor, model, device: str = 'cpu',
                     sample_rate: int = 16000):
    """
    Extract Wav2Vec2-base hidden-state features from an audio file.

    Thay cho MFCC + chromagram: dùng `wav2vec_feature_extractor` để lấy
    `last_hidden_state` (time_len, 768) từ facebook/wav2vec2-base.

    Args:
        file_path: Path to the audio file.
        processor: Wav2Vec2Processor đã load (xem `load_wav2vec2`).
        model: Wav2Vec2Model đã load (xem `load_wav2vec2`).
        device: 'cpu' hoặc 'cuda'.
        sample_rate: Wav2Vec2-base yêu cầu 16kHz.

    Returns:
        features: (time_len, 768) feature array, hoặc None on error.
    """
    from wav2vec_feature_extractor import extract_wav2vec2_features

    return extract_wav2vec2_features(
        file_path=file_path, processor=processor, model=model,
        device=device, target_sample_rate=sample_rate
    )


### Wav2Vec2 Feature Extraction (single pass, không cần global normalization)

In [6]:
def process_dataset(dataset_path: str,
                    x_output_path_template: str,
                    y_output_path_template: str,
                    id_output_path_template: str,
                    wav2vec2_model_name: str,
                    sample_rate: int,
                    device: str):
    """
    Single-pass Wav2Vec2 feature extraction (không cần global mean/std).

    Khác với pipeline MFCC cũ (2 lượt: tính thống kê toàn cục rồi chuẩn hoá),
    hidden states của Wav2Vec2 đã có scale ổn định sẵn, và LayerNorm trong
    `ResLSTM_Multi_Att.forward` sẽ đảm nhiệm việc chuẩn hoá trước khi vào LSTM.
    Vì vậy chỉ cần một lượt: trích đặc trưng rồi lưu thẳng thành .npy.
    """
    from wav2vec_feature_extractor import load_wav2vec2

    valid_files = []
    valid_emotions = []
    valid_actors = []

    emotions_dict = get_emotions_dictionary()
    actors_dict = get_actors()
    observed_emotions = get_emotions()

    all_files = list(glob.glob(dataset_path))
    if not all_files:
        raise FileNotFoundError(
            f"No audio files found at {dataset_path}. "
            f"Please download the RAVDESS dataset first."
        )

    print(f"Loading {wav2vec2_model_name}...")
    processor, model = load_wav2vec2(wav2vec2_model_name, device=device)

    os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)

    print("Extracting Wav2Vec2 features...")
    idx = 0
    for file in tqdm(all_files, desc="Extracting"):
        file_name = os.path.basename(file)
        splitted = file_name.split("-")
        emotion = emotions_dict[splitted[2]]

        if emotion not in observed_emotions:
            continue

        actor = actors_dict[splitted[6].split(".")[0]]
        features = extract_features(file_path=file, processor=processor, model=model,
                                    device=device, sample_rate=sample_rate)

        if features is None:
            continue

        x_path = x_output_path_template.format(idx)
        np.save(x_path, features)

        valid_files.append(file)
        valid_emotions.append(emotion)
        valid_actors.append(actor)
        idx += 1

    if not valid_files:
        raise RuntimeError("No valid data found.")

    y_path = y_output_path_template
    np.save(y_path, np.array(valid_emotions))

    id_path = id_output_path_template
    np.save(id_path, np.array(valid_actors))

    print(f"Processed {len(valid_files)} files.")


### Run Feature Extraction

In [7]:
WAV2VEC2_MODEL_NAME = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000

os.makedirs(EXTRACTED_DATA_DIR, exist_ok=True)

x_template = os.path.join(EXTRACTED_DATA_DIR, "X_{}_wav2vec2.npy")
y_path = os.path.join(EXTRACTED_DATA_DIR, "y_labels_wav2vec2.npy")
id_path = os.path.join(EXTRACTED_DATA_DIR, "IDs_wav2vec2.npy")

if not os.path.exists(y_path):
    process_dataset(
        dataset_path=RAVDESS_DATA_PATH,
        x_output_path_template=x_template,
        y_output_path_template=y_path,
        id_output_path_template=id_path,
        wav2vec2_model_name=WAV2VEC2_MODEL_NAME,
        sample_rate=SAMPLE_RATE,
        device=str(DEVICE)
    )
else:
    print(f"Features already extracted at {EXTRACTED_DATA_DIR}")


Features already extracted at d:\Speech_Processing\ResLSTM-SER_with_wav2vec\data\ravdess\extracted


## PyTorch Dataset

In [8]:
class RavdessAudioDataset(Dataset):
    """RAVDESS dataset for variable-length sequences with speaker-based folds."""

    def __init__(self,
                 mfccs_dir: str,
                 annotations_file: str,
                 actor_ids_file: str,
                 desired_labels: list = None):
        self.mfccs_dir = mfccs_dir
        self.label_mapping = {
            'neutral': 0, 'calm': 1, 'happy': 2, 'sad': 3,
            'angry': 4, 'fear': 5, 'disgust': 6, 'surprised': 7
        }

        y = np.load(annotations_file, allow_pickle=True)
        ids = np.load(actor_ids_file, allow_pickle=True)

        if desired_labels is not None:
            mask = np.isin(y, desired_labels)
            y = y[mask]
            ids = ids[mask]

        self.y = torch.tensor([self.label_mapping[label] for label in y], dtype=torch.long)
        self.X_ids = [int(a) for a in ids]

        self.feature_files = []
        for i in range(len(self.y)):
            fpath = mfccs_dir.format(i)
            self.feature_files.append(fpath)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        features = np.load(self.feature_files[index])
        features = torch.tensor(features, dtype=torch.float32)
        label = self.y[index]
        return features, label

    def get_kth_fold_inds(self, fold_num: int):
        """
        Speaker-independent 5-fold split (same as paper).
        Returns (train_indices, val_indices, test_indices).
        """
        folds = [
            [2, 5, 14, 15, 16],
            [3, 6, 7, 13, 18],
            [10, 11, 12, 19, 20],
            [8, 17, 21, 23, 24],
            [1, 4, 9, 22],
        ]
        folds_val = [
            [3, 6],
            [10, 11],
            [8, 17],
            [1, 14],
            [2, 5],
        ]

        ids_train, ids_val, ids_test = [], [], []
        for i in range(len(self.X_ids)):
            if self.X_ids[i] in folds[fold_num]:
                ids_test.append(i)
            elif self.X_ids[i] in folds_val[fold_num]:
                ids_val.append(i)
            else:
                ids_train.append(i)
        return ids_train, ids_val, ids_test


def pad_mfcc_sequence(batch: List[Tuple[torch.Tensor, int]]):
    """Collate function: pad sequences and return (padded, labels, lengths).

    Tên hàm giữ nguyên (pad_mfcc_sequence) dù giờ dùng cho đặc trưng
    Wav2Vec2, để không phải sửa lời gọi collate_fn bên trong k_fold_cv.
    """
    sequences, labels = zip(*batch)
    lengths = torch.tensor([seq.size(0) for seq in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True)
    labels = torch.stack(labels) if isinstance(labels[0], torch.Tensor) else torch.tensor(labels, dtype=torch.long)
    return padded, labels, lengths


### Create Dataset Instance

In [9]:
feature_dir_template = os.path.join(EXTRACTED_DATA_DIR, "X_{}_wav2vec2.npy")

dataset = RavdessAudioDataset(
    mfccs_dir=feature_dir_template,
    annotations_file=y_path,
    actor_ids_file=id_path,
    desired_labels=get_emotions()
)
print(f"Dataset: {len(dataset)} samples, 8 classes")


Dataset: 1440 samples, 8 classes


## Models

### ResLSTM-Wav2Vec (import từ res_lstm.py)

In [10]:
# Kiến trúc model (ResLSTM + Multi-Vector Attention + Projection Layer cho
# Wav2Vec2) được định nghĩa trong res_lstm.py — đây là "authoritative source",
# notebook chỉ import để tránh định nghĩa trùng lặp.
from src.models.res_lstm import ResLSTM_Multi_Att, MODEL_VERSION

print(f"Model version: {MODEL_VERSION}")


Model version: v2


## Evaluation Utilities

In [11]:
from ignite.engine import Engine

def _eval_step(engine, batch):
    return batch

def get_default_evaluator():
    return Engine(_eval_step)

def get_test_evaluator():
    return Engine(_eval_step)

## Training Loop

In [12]:
from pytorch_metric_learning import losses, miners
import numpy as np
import torch
from ignite.metrics.recall import Recall

def get_current_alpha(epoch, warmup_epochs=15, target_alpha=0.5):
    if epoch < warmup_epochs:
        return target_alpha * (epoch / warmup_epochs)
    return target_alpha

def training_loop(model, loss_fn_ce, loss_fn_triplet, miner, target_alpha, optimizer, lr_scheduler,
                  train_loader, val_loader, n_epochs, model_path,
                  grad_clip_norm=5.0, early_stopping_patience=20):
    """
    ĐÃ SỬA (so với bản gốc):
    - Thêm gradient clipping (grad_clip_norm) để tránh exploding gradient của LSTM,
      áp dụng đúng cách kể cả khi dùng AMP (unscale_ trước khi clip).
    - Thêm early stopping (early_stopping_patience) dựa trên UAR validation để giảm
      overfitting và tiết kiệm thời gian huấn luyện trên tập RAVDESS nhỏ.
    Chữ ký hàm giữ nguyên tên, chỉ thêm 2 tham số mới có giá trị mặc định
    -> không phá vỡ các lệnh gọi training_loop(...) hiện có trong notebook.
    """
    best_epoch = -1
    UAR_val_max = -1
    epochs_no_improve = 0
    loss_train_history = np.zeros(n_epochs)
    loss_val_history = np.zeros(n_epochs)

    scaler = torch.amp.GradScaler('cuda') if torch.cuda.is_available() else None
    default_evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(default_evaluator, "macro_recall")

    for epoch in range(n_epochs):
        current_alpha = get_current_alpha(epoch, target_alpha=target_alpha)

        model.train()
        loss_train = 0.0
        for X, labels, lengths in train_loader:
            X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
            optimizer.zero_grad()

            if scaler is not None:
                with torch.amp.autocast('cuda'):
                    logits, embeddings = model(X, lengths, return_embeddings=True)
                    loss_ce = loss_fn_ce(logits, labels)

                    hard_pairs = miner(embeddings, labels)
                    loss_trip = loss_fn_triplet(embeddings, labels, hard_pairs)

                    loss = loss_ce + (current_alpha * loss_trip)

                scaler.scale(loss).backward()
                # Gradient clipping: phải unscale_ trước khi clip khi dùng AMP,
                # nếu không grad_norm sẽ bị tính sai theo scale factor.
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits, embeddings = model(X, lengths, return_embeddings=True)
                loss_ce = loss_fn_ce(logits, labels)
                hard_pairs = miner(embeddings, labels)
                loss_trip = loss_fn_triplet(embeddings, labels, hard_pairs)
                loss = loss_ce + (current_alpha * loss_trip)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip_norm)
                optimizer.step()

            loss_train += loss.item()
        loss_train /= len(train_loader)

        model.eval()
        loss_val = 0.0
        val_pred_all, val_true_all = [], []
        with torch.no_grad():
            for X, labels, lengths in val_loader:
                X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                logits = model(X, lengths)
                loss = loss_fn_ce(logits, labels)
                loss_val += loss.item()
                val_pred_all.append(torch.softmax(logits, dim=1))
                val_true_all.append(labels)

        val_pred_all = torch.cat(val_pred_all)
        val_true_all = torch.cat(val_true_all)
        loss_val /= len(val_loader)
        loss_train_history[epoch] = loss_train
        loss_val_history[epoch] = loss_val

        default_evaluator.terminate()
        state = default_evaluator.run([[val_pred_all, val_true_all]])
        UAR_val = state.metrics["macro_recall"]

        if lr_scheduler is not None:
            lr_scheduler.step()

        if UAR_val > UAR_val_max:
            UAR_val_max = UAR_val
            best_epoch = epoch
            epochs_no_improve = 0
            torch.save(model.state_dict(), model_path)
        else:
            epochs_no_improve += 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch + 1}/{n_epochs} [Alpha: {current_alpha:.2f}] | "
                  f"Train loss: {loss_train:.4f} | Val loss: {loss_val:.4f} | Val UAR: {UAR_val:.4f}")

        # Early stopping: dừng nếu UAR validation không cải thiện sau
        # `early_stopping_patience` epoch liên tiếp (giảm overfitting + tiết kiệm compute).
        if epochs_no_improve >= early_stopping_patience:
            print(f"Early stopping tại epoch {epoch + 1} (không cải thiện sau {early_stopping_patience} epoch).")
            break

    model.load_state_dict(torch.load(model_path, weights_only=True))
    return loss_train_history, loss_val_history, best_epoch, UAR_val_max


## K-Fold Cross-Validation

In [13]:
from pytorch_metric_learning.samplers import MPerClassSampler

def k_fold_cv(dataset, model_factory, loss_fn_ce, loss_fn_triplet, miner, alpha, optimizer_factory,
              lr_scheduler_factory, n_epochs, k_fold, batch_size,
              init_model_path, model_path):
    oof_preds, oof_targets = [], []

    for fold in range(k_fold):
        print(f"\n{'='*60}")
        print(f"Fold {fold + 1}/{k_fold}")
        print(f"{'='*60}")

        inds_train, inds_val, inds_test = dataset.get_kth_fold_inds(fold)
        train_set = torch.utils.data.Subset(dataset, inds_train)
        val_set = torch.utils.data.Subset(dataset, inds_val)
        test_set = torch.utils.data.Subset(dataset, inds_test)

        # --- ĐÃ SỬA: Tích hợp Sampler cho Dataloader để hỗ trợ Triplet Loss ---
        train_labels = dataset.y[train_set.indices]
        # Ép mỗi batch có ít nhất 2 đến 4 mẫu cùng class tùy theo batch_size
        m_per_class = max(2, batch_size // NUM_CLASSES) if batch_size >= NUM_CLASSES else 2 
        
        sampler = MPerClassSampler(
            labels=train_labels,
            m=m_per_class,
            batch_size=batch_size,
            length_before_new_iter=len(train_set)
        )

        train_loader = torch.utils.data.DataLoader(
            train_set, 
            batch_size=batch_size, 
            sampler=sampler,              # Dùng Custom Sampler
            collate_fn=pad_mfcc_sequence,
            num_workers=0, 
            pin_memory=True
            # shuffle=True ĐÃ BỊ XÓA vì xung đột với custom sampler
        )

        val_loader = torch.utils.data.DataLoader(
            val_set, batch_size=batch_size, shuffle=False, collate_fn=pad_mfcc_sequence,
            num_workers=0, pin_memory=True)
        test_loader = torch.utils.data.DataLoader(
            test_set, batch_size=batch_size, shuffle=False, collate_fn=pad_mfcc_sequence,
            num_workers=0, pin_memory=True)

        set_seed(42 + fold * 100)
        model = model_factory()
        torch.save(model.state_dict(), init_model_path)
        optimizer = optimizer_factory(model)
        lr_scheduler = lr_scheduler_factory(optimizer)

        training_loop(model, loss_fn_ce, loss_fn_triplet, miner, alpha, optimizer, lr_scheduler,
                      train_loader, val_loader, n_epochs, model_path)

        model.eval()
        fold_preds, fold_targets = [], []
        with torch.no_grad():
            for X, labels, lengths in test_loader:
                X, labels, lengths = X.to(DEVICE), labels.to(DEVICE), lengths.to(DEVICE)
                logits = model(X, lengths)
                fold_preds.append(torch.softmax(logits, dim=1).cpu())
                fold_targets.append(labels.cpu())
        oof_preds.append(torch.cat(fold_preds))
        oof_targets.append(torch.cat(fold_targets))

    return oof_preds, oof_targets

## Hyperparameter Optimization with Optuna

### Configure Search Space

In [14]:
INPUT_SIZE = 768        # dim của Wav2Vec2-base last_hidden_state
PROJECTION_DIM = 256    # Projection Layer: 768 -> 256 (input/hidden của LSTM1)
NUM_CLASSES = 8
NUM_LAYERS = 1
NUM_ATT = 4             # số vector trong Multi-Vector Attention
N_EPOCHS = 100
K_FOLD = 5

# LSTM2 hidden size (embedding dim)
HIDDEN_SIZE = 512

SQLITE_STORAGE_URL = f"sqlite:///optuna_h{HIDDEN_SIZE}_wav2vec_triplet.db"
STUDY_NAME = f"ResLSTM_Wav2Vec_H{HIDDEN_SIZE}_Triplet_tuning_v2"

print(f"Hidden size: {HIDDEN_SIZE}, Study: {STUDY_NAME}")


Hidden size: 512, Study: ResLSTM_Wav2Vec_H512_Triplet_tuning_v2


### Objective Function

In [15]:
from pytorch_metric_learning import losses, miners

def objective(trial: optuna.Trial) -> float:
    lr = trial.suggest_float("lr", 1e-5, 3e-2, log=True)

    # Loại bỏ batch size 4 và 8 để Miner có thể tìm được cặp Positive/Negative
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    dropout_rate = trial.suggest_float("dropout_rate", 0.02, 0.5)
    weight_decay = trial.suggest_float("wd", 2e-5, 4e-2, log=True)
    t_0_loop = trial.suggest_categorical("t_0_loop", [1, 2, 3, 5, 10])

    # ĐÃ SỬA: target_alpha (hệ số Triplet) và margin giờ được Optuna search thay vì
    # hard-code, vì ảnh hưởng trực tiếp đến mức độ regularize/overfit của embedding.
    target_alpha = trial.suggest_float("target_alpha", 0.1, 1.0)
    margin = trial.suggest_float("margin", 0.1, 0.5)
    # ĐÃ SỬA: label smoothing giúp giảm overconfidence/overfitting của CE trên tập nhỏ.
    label_smoothing = trial.suggest_float("label_smoothing", 0.0, 0.15)

    set_seed(42)

    def model_factory():
        return ResLSTM_Multi_Att(
            input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, num_att=NUM_ATT, num_classes=NUM_CLASSES,
            projection_dim=PROJECTION_DIM,
            dropout_p=dropout_rate, device=str(DEVICE)
        ).to(DEVICE)

    def optimizer_factory(model):
        # ĐÃ SỬA: AdamW thay vì Adam -> weight decay decoupled khỏi gradient adaptive,
        # đúng chuẩn hơn khi weight_decay được search tới 4e-2 (Adam cổ điển sẽ làm
        # weight decay tương tác sai với learning rate/adaptive moment).
        return optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    def scheduler_factory(optimizer):
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=t_0_loop, T_mult=1, eta_min=lr * 0.01)

    loss_fn_ce = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    loss_fn_triplet = losses.TripletMarginLoss(margin=margin)
    miner = miners.MultiSimilarityMiner()

    os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)
    init_model_path = os.path.join(MODEL_BACKUP_DIR, f"init_{STUDY_NAME}.pt")
    model_path = os.path.join(MODEL_BACKUP_DIR, f"best_{STUDY_NAME}.pt")

    oof_preds, oof_targets = k_fold_cv(
        dataset=dataset, model_factory=model_factory,
        loss_fn_ce=loss_fn_ce, loss_fn_triplet=loss_fn_triplet, miner=miner, alpha=target_alpha,
        optimizer_factory=optimizer_factory, lr_scheduler_factory=scheduler_factory,
        n_epochs=N_EPOCHS, k_fold=K_FOLD, batch_size=batch_size,
        init_model_path=init_model_path, model_path=model_path)

    all_preds = torch.cat([p for p in oof_preds])
    all_targets = torch.cat([t for t in oof_targets])

    evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(evaluator, "macro_recall")
    evaluator.terminate()
    state = evaluator.run([[all_preds, all_targets]])
    return state.metrics["macro_recall"]


### Run Optuna Study

In [16]:
N_TRIALS = 50  # Bạn có thể giảm xuống 50 nếu muốn test nhanh

study = optuna.create_study(
    storage=SQLITE_STORAGE_URL,
    study_name=STUDY_NAME,
    direction="maximize",
    load_if_exists=True,
)

print(f"Starting Optuna optimization with {N_TRIALS} trials...")

try:
    # Hàm optimize sẽ tự động gọi hàm objective Multi-task bạn vừa sửa
    study.optimize(objective, n_trials=N_TRIALS)
except KeyboardInterrupt:
    # Bẫy lỗi: Dừng êm đẹp nếu bạn chủ động bấm Stop (Interrupt Kernel)
    print("\nQuá trình tối ưu hóa đã được người dùng dừng lại ngang chừng.")

print("\nBest hyperparameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")
print(f"  Best UAR: {study.best_value:.4f}")

[I 2026-07-10 03:23:50,549] A new study created in RDB with name: ResLSTM_Wav2Vec_H512_Triplet_tuning_v2


Starting Optuna optimization with 50 trials...

Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8939 | Val loss: 1.9312 | Val UAR: 0.3359
Epoch 10/100 [Alpha: 0.40] | Train loss: 1.3368 | Val loss: 1.6893 | Val UAR: 0.3594
Epoch 20/100 [Alpha: 0.67] | Train loss: 1.0970 | Val loss: 1.5253 | Val UAR: 0.5078
Epoch 30/100 [Alpha: 0.67] | Train loss: 0.7665 | Val loss: 1.4387 | Val UAR: 0.5703
Epoch 40/100 [Alpha: 0.67] | Train loss: 0.7851 | Val loss: 1.4282 | Val UAR: 0.6641
Early stopping tại epoch 45 (không cải thiện sau 20 epoch).

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8748 | Val loss: 1.8481 | Val UAR: 0.4453
Epoch 10/100 [Alpha: 0.40] | Train loss: 1.3176 | Val loss: 1.8325 | Val UAR: 0.3750
Epoch 20/100 [Alpha: 0.67] | Train loss: 1.0634 | Val loss: 1.5523 | Val UAR: 0.5234
Epoch 30/100 [Alpha: 0.67] | Train loss: 0.7654 | Val loss: 1.6167 | Val UAR: 0.5156
Early stopping tại epoch 32 (không cải thiện sau 20 epoch).

Fold 3/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 

[I 2026-07-10 03:36:19,281] Trial 0 finished with value: 0.5546875 and parameters: {'lr': 0.00012355671798121068, 'batch_size': 16, 'dropout_rate': 0.09523757285504339, 'wd': 0.0020199263537244174, 't_0_loop': 3, 'target_alpha': 0.6693120893677736, 'margin': 0.10835598855448399, 'label_smoothing': 0.11279926422855491}. Best is trial 0 with value: 0.5546875.



Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8975 | Val loss: 2.0266 | Val UAR: 0.2812
Epoch 10/100 [Alpha: 0.12] | Train loss: 1.0492 | Val loss: 1.4113 | Val UAR: 0.6250
Epoch 20/100 [Alpha: 0.20] | Train loss: 0.9055 | Val loss: 1.5339 | Val UAR: 0.5625
Epoch 30/100 [Alpha: 0.20] | Train loss: 0.7456 | Val loss: 1.4966 | Val UAR: 0.6562
Epoch 40/100 [Alpha: 0.20] | Train loss: 0.7027 | Val loss: 1.7124 | Val UAR: 0.5859
Epoch 50/100 [Alpha: 0.20] | Train loss: 0.7332 | Val loss: 1.6675 | Val UAR: 0.5547
Epoch 60/100 [Alpha: 0.20] | Train loss: 0.6579 | Val loss: 1.6270 | Val UAR: 0.5859
Epoch 70/100 [Alpha: 0.20] | Train loss: 0.6486 | Val loss: 1.5058 | Val UAR: 0.7109
Epoch 80/100 [Alpha: 0.20] | Train loss: 0.6617 | Val loss: 1.6766 | Val UAR: 0.5938
Epoch 90/100 [Alpha: 0.20] | Train loss: 0.6397 | Val loss: 1.6356 | Val UAR: 0.6172
Early stopping tại epoch 90 (không cải thiện sau 20 epoch).

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8414 | Val loss: 1.8317 | Val

[I 2026-07-10 03:49:39,646] Trial 1 finished with value: 0.5729166666666666 and parameters: {'lr': 0.00152015607739474, 'batch_size': 16, 'dropout_rate': 0.28004806165894064, 'wd': 0.00011194094769198271, 't_0_loop': 1, 'target_alpha': 0.2021151962584598, 'margin': 0.38203986611211727, 'label_smoothing': 0.13127506454101948}. Best is trial 1 with value: 0.5729166666666666.


...


Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.7713 | Val loss: 1.8638 | Val UAR: 0.4297
Epoch 10/100 [Alpha: 0.49] | Train loss: 1.0404 | Val loss: 1.5454 | Val UAR: 0.5000
Epoch 20/100 [Alpha: 0.82] | Train loss: 0.8018 | Val loss: 1.7021 | Val UAR: 0.4766
Epoch 30/100 [Alpha: 0.82] | Train loss: 0.6051 | Val loss: 1.4466 | Val UAR: 0.6641
Epoch 40/100 [Alpha: 0.82] | Train loss: 0.5676 | Val loss: 1.6155 | Val UAR: 0.5938
Early stopping tại epoch 47 (không cải thiện sau 20 epoch).

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.7809 | Val loss: 1.8344 | Val UAR: 0.2891
Epoch 10/100 [Alpha: 0.49] | Train loss: 1.0208 | Val loss: 1.4988 | Val UAR: 0.5312
Epoch 20/100 [Alpha: 0.82] | Train loss: 0.7939 | Val loss: 1.7498 | Val UAR: 0.4297
Epoch 30/100 [Alpha: 0.82] | Train loss: 0.6269 | Val loss: 1.6545 | Val UAR: 0.5078
Epoch 40/100 [Alpha: 0.82] | Train loss: 0.5465 | Val loss: 1.6964 | Val UAR: 0.5469
Epoch 50/100 [Alpha: 0.82] | Train loss: 0.5465 | Val loss: 1.8776 | Val

[I 2026-07-10 11:45:30,288] Trial 48 finished with value: 0.5481770833333334 and parameters: {'lr': 0.0009144301955257343, 'batch_size': 16, 'dropout_rate': 0.13206600696469523, 'wd': 5.9811814163397115e-05, 't_0_loop': 1, 'target_alpha': 0.8154291212877263, 'margin': 0.2410561852283195, 'label_smoothing': 0.08891903538873781}. Best is trial 1 with value: 0.5729166666666666.



Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 2.0120 | Val loss: 1.9973 | Val UAR: 0.3828
Epoch 10/100 [Alpha: 0.45] | Train loss: 1.6150 | Val loss: 1.6317 | Val UAR: 0.5234
Epoch 20/100 [Alpha: 0.75] | Train loss: 1.5173 | Val loss: 1.5808 | Val UAR: 0.5625
Epoch 30/100 [Alpha: 0.75] | Train loss: 1.3745 | Val loss: 1.5616 | Val UAR: 0.5938
Early stopping tại epoch 38 (không cải thiện sau 20 epoch).

Fold 2/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.9930 | Val loss: 1.9667 | Val UAR: 0.3047
Epoch 10/100 [Alpha: 0.45] | Train loss: 1.5632 | Val loss: 1.7149 | Val UAR: 0.4609
Epoch 20/100 [Alpha: 0.75] | Train loss: 1.4712 | Val loss: 1.6670 | Val UAR: 0.5078
Epoch 30/100 [Alpha: 0.75] | Train loss: 1.3220 | Val loss: 1.6314 | Val UAR: 0.5000
Epoch 40/100 [Alpha: 0.75] | Train loss: 1.1213 | Val loss: 1.6684 | Val UAR: 0.4609
Epoch 50/100 [Alpha: 0.75] | Train loss: 0.9513 | Val loss: 1.6800 | Val UAR: 0.4609
Early stopping tại epoch 51 (không cải thiện sau 20 epoch).

Fold 3/5
Ep

[I 2026-07-10 11:51:20,374] Trial 49 finished with value: 0.53125 and parameters: {'lr': 5.2341729606312275e-05, 'batch_size': 32, 'dropout_rate': 0.2136586445603012, 'wd': 0.00014534704171713855, 't_0_loop': 1, 'target_alpha': 0.7529439480420015, 'margin': 0.2282285857110627, 'label_smoothing': 0.13550642745398478}. Best is trial 1 with value: 0.5729166666666666.



Best hyperparameters:
  lr: 0.00152015607739474
  batch_size: 16
  dropout_rate: 0.28004806165894064
  wd: 0.00011194094769198271
  t_0_loop: 1
  target_alpha: 0.2021151962584598
  margin: 0.38203986611211727
  label_smoothing: 0.13127506454101948
  Best UAR: 0.5729


In [28]:
import optuna
import pandas as pd

# 1. Load chính xác study 'v2'
study_name = "ResLSTM_Wav2Vec_H512_Triplet_tuning_v2"
storage = "sqlite:///optuna_h512_wav2vec_triplet.db"

try:
    study = optuna.load_study(study_name=study_name, storage=storage)
    print(f"✅ Đã load thành công study: {study_name}")
    
    # 2. Hiển thị thông tin tổng quan
    print("-" * 30)
    print("🏆 BỘ SIÊU THAM SỐ TỐT NHẤT (Best Params):")
    for key, value in study.best_params.items():
        print(f" - {key}: {value}")
    
    print("-" * 30)
    print(f"📊 Độ chính xác (UAR) cao nhất: {study.best_value:.4f}")
    print(f"📈 Tổng số Trials đã chạy: {len(study.trials)}")
    
    # 3. Hiển thị dưới dạng DataFrame để dễ quan sát (Tùy chọn)
    print("-" * 30)
    print("📋 Bảng 5 Trial tốt nhất:")
    df = study.trials_dataframe()
    # Sắp xếp theo giá trị tốt nhất (giả sử UAR càng cao càng tốt)
    best_trials = df.sort_values(by="value", ascending=False).head(5)
    print(best_trials[['number', 'value', 'params_lr', 'params_batch_size']])

except Exception as e:
    print(f"❌ Lỗi khi load study: {e}")

✅ Đã load thành công study: ResLSTM_Wav2Vec_H512_Triplet_tuning_v2
------------------------------
🏆 BỘ SIÊU THAM SỐ TỐT NHẤT (Best Params):
 - lr: 0.00152015607739474
 - batch_size: 16
 - dropout_rate: 0.28004806165894064
 - wd: 0.00011194094769198271
 - t_0_loop: 1
 - target_alpha: 0.2021151962584598
 - margin: 0.38203986611211727
 - label_smoothing: 0.13127506454101948
------------------------------
📊 Độ chính xác (UAR) cao nhất: 0.5729
📈 Tổng số Trials đã chạy: 50
------------------------------
📋 Bảng 5 Trial tốt nhất:
    number     value  params_lr  params_batch_size
1        1  0.572917   0.001520                 16
23      23  0.569661   0.001644                 16
42      42  0.564453   0.000162                 16
33      33  0.559896   0.000575                 64
44      44  0.558594   0.000196                 16


In [23]:
import sqlite3
conn = sqlite3.connect("optuna_h512_wav2vec_triplet.db")
cursor = conn.cursor()
cursor.execute("SELECT study_name FROM studies")
print("Các Study hiện có trong DB:", cursor.fetchall())
conn.close()

Các Study hiện có trong DB: [('ResLSTM_Wav2Vec_H512_Triplet_tuning_v1',), ('ResLSTM_Wav2Vec_H512_Triplet_tuning_v2',)]


In [29]:
from optuna.visualization import plot_optimization_history, plot_param_importances

# Biểu đồ 1: Xem quá trình UAR tăng dần qua các trials
fig1 = plot_optimization_history(study)
fig1.show()

# Biểu đồ 2: Đánh giá độ quan trọng của từng tham số (Ví dụ: margin hay learning rate quan trọng hơn?)
fig2 = plot_param_importances(study)
fig2.show()

## 10-Run Statistical Evaluation

Run 10 independent training cycles with the best hyperparameters to get mean ± std UAR.

In [30]:
import os
from pytorch_metric_learning import losses, miners

os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

# Load best params from Optuna
study = optuna.load_study(study_name=STUDY_NAME, storage=SQLITE_STORAGE_URL)
best_params = study.best_params
print("Using hyperparameters:")
for k, v in best_params.items():
    print(f"  {k}: {v}")

results = []
for run in range(10):
    print(f"\n{'='*60}")
    print(f"Run {run + 1}/10")
    print(f"{'='*60}")

    set_seed(42 + run * 100)

    def model_factory():
        return ResLSTM_Multi_Att(
            input_size=INPUT_SIZE, hidden_size=HIDDEN_SIZE,
            num_layers=NUM_LAYERS, num_att=NUM_ATT, num_classes=NUM_CLASSES,
            projection_dim=PROJECTION_DIM,
            dropout_p=best_params['dropout_rate'], device=str(DEVICE)
        ).to(DEVICE)

    def optimizer_factory(model):
        # ĐÃ SỬA: AdamW đồng bộ với objective()
        return optim.AdamW(model.parameters(), lr=best_params['lr'],
                           weight_decay=best_params['wd'])

    def scheduler_factory(optimizer):
        return optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=best_params['t_0_loop'], T_mult=1,
            eta_min=best_params['lr'] * 0.01)

    # ĐÃ SỬA: margin/target_alpha/label_smoothing lấy từ best_params (đã search ở Optuna)
    # thay vì hard-code, dùng .get(...) để vẫn chạy được với study cũ chưa có các key này.
    loss_fn_ce = nn.CrossEntropyLoss(label_smoothing=best_params.get('label_smoothing', 0.0))
    loss_fn_triplet = losses.TripletMarginLoss(margin=best_params.get('margin', 0.3))
    miner = miners.MultiSimilarityMiner()
    target_alpha = best_params.get('target_alpha', 0.5)

    model_name = f"ResLSTM_Wav2Vec_H{HIDDEN_SIZE}"
    init_model_path = os.path.join(MODEL_BACKUP_DIR, f"init_{model_name}_run{run}.pt")
    model_path = os.path.join(MODEL_BACKUP_DIR, f"best_{model_name}_run{run}.pt")

    oof_preds, oof_targets = k_fold_cv(
        dataset=dataset, model_factory=model_factory,
        loss_fn_ce=loss_fn_ce, loss_fn_triplet=loss_fn_triplet, miner=miner, alpha=target_alpha,
        optimizer_factory=optimizer_factory, lr_scheduler_factory=scheduler_factory,
        n_epochs=N_EPOCHS, k_fold=K_FOLD, batch_size=best_params['batch_size'],
        init_model_path=init_model_path, model_path=model_path)

    all_preds = torch.cat([p for p in oof_preds])
    all_targets = torch.cat([t for t in oof_targets])

    evaluator = get_default_evaluator()
    macro_metric = Recall(average=True)
    macro_metric.attach(evaluator, "macro_recall")
    evaluator.terminate()
    state = evaluator.run([[all_preds, all_targets]])
    uar = state.metrics["macro_recall"]

    _m = model_factory()
    num_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)

    results.append({"run": run + 1, "uar": uar, "num_params": num_params})
    print(f"Run {run + 1}: UAR = {uar:.4f}, Params = {num_params}")

# Statistics
uars = [r["uar"] for r in results]
mean_uar = np.mean(uars)
std_uar = np.std(uars)
print(f"\nFinal: UAR = {mean_uar:.4f} +/- {std_uar:.4f}")
print(f"Num params: {results[0]['num_params']}")


Using hyperparameters:
  lr: 0.00152015607739474
  batch_size: 16
  dropout_rate: 0.28004806165894064
  wd: 0.00011194094769198271
  t_0_loop: 1
  target_alpha: 0.2021151962584598
  margin: 0.38203986611211727
  label_smoothing: 0.13127506454101948

Run 1/10

Fold 1/5
Epoch 1/100 [Alpha: 0.00] | Train loss: 1.8975 | Val loss: 2.0266 | Val UAR: 0.2812
Epoch 10/100 [Alpha: 0.12] | Train loss: 1.0492 | Val loss: 1.4113 | Val UAR: 0.6250
Epoch 20/100 [Alpha: 0.20] | Train loss: 0.9055 | Val loss: 1.5339 | Val UAR: 0.5625
Epoch 30/100 [Alpha: 0.20] | Train loss: 0.7456 | Val loss: 1.4966 | Val UAR: 0.6562
Epoch 40/100 [Alpha: 0.20] | Train loss: 0.7027 | Val loss: 1.7124 | Val UAR: 0.5859
Epoch 50/100 [Alpha: 0.20] | Train loss: 0.7332 | Val loss: 1.6675 | Val UAR: 0.5547
Epoch 60/100 [Alpha: 0.20] | Train loss: 0.6579 | Val loss: 1.6270 | Val UAR: 0.5859
Epoch 70/100 [Alpha: 0.20] | Train loss: 0.6486 | Val loss: 1.5058 | Val UAR: 0.7109
Epoch 80/100 [Alpha: 0.20] | Train loss: 0.6617 | Va

## Summary

This notebook reproduces the full experimental pipeline from the DSPA 2026 paper:

1. **Feature Extraction**: 34 MFCC + 12 chroma = 46-dim features, two-pass global normalization
2. **Model**: ResLSTM-SA — residual LSTM with soft attention
3. **Hyperparameter Tuning**: Bayesian optimization with Optuna (TPE sampler)
4. **Evaluation**: 10 independent runs, speaker-independent 5-fold CV, reporting UAR mean ± std

### Key Results (from paper)

| Model | Hidden Size | Params | UAR (mean ± std) | Max UAR |
|-------|-------------|--------|------------------|---------|
| ResLSTM-SA-h64 | 64 | 46.8k | 0.6232 ± 0.0119 | **0.6517** |
| ResLSTM-SA-h128 | 128 | 108.9k | 0.6107 ± 0.0134 | 0.6348 |
| ResLSTM-SA-h32 | 32 | 28.0k | 0.6130 ± 0.0111 | 0.6315 |

### References

- Krasnoproshin, D. & Vashkevich, M. (2026). *Speech Emotion Recognition using Attention-based LSTM-Network with Residual Connection.* DSPA 2026.
- Mirsamadi, S., Barsoum, E., & Zhang, C. (2017). *Automatic speech emotion recognition using recurrent neural networks with local attention.* ICASSP 2017.
- Livingstone, S. R. & Russo, F. A. (2018). *The RAVDESS database.* PLoS ONE.
- Akiba, T. et al. (2019). *Optuna: A Next-generation Hyperparameter Optimization Framework.* KDD 2019.